# Deledalle, Tupin, Denis (2010) — Poisson NL Means / 포아송 비지역 평균

**Paper**: Deledalle, C.-A., Tupin, F., & Denis, L. (2010). *Proc. IEEE ICIP*, pp. 801-804. [DOI: 10.1109/ICIP.2010.5653394]

## Overview / 개요

이 노트북은 Poisson NL Means의 핵심 요소를 구현하고 검증한다:
1. **Poisson likelihood-ratio patch distance** $f_L$ (Eq. 5)와 Euclidean 거리의 비교
2. **Symmetric KL divergence** $g_{KL}$ (Eq. 6) 사전 추정 패치 거리 
3. **Refined NL Means with two parameters** ($\alpha, \beta$) — 잡음 영상 + 사전 추정 영상 동시 활용
4. **PURE (Eq. 9)** for unbiased MSE estimation via Chen's identity
5. **저광자 영역에서 classical NL Means와 비교** — Poisson NL Means가 더 강건함을 시연

Implements the core ingredients of Poisson NL Means: the likelihood-ratio patch distance, KL divergence on pre-estimates, refined NL Means with two parameters, PURE estimator via Chen's identity, and a comparison with classical Euclidean NL Means at low photon counts.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import NDArray
from scipy.ndimage import uniform_filter
from skimage import data, img_as_float

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True

rng = np.random.default_rng(2026)

---

## Part 1: Patch distances — Poisson likelihood vs Euclidean / 패치 거리 비교

Eq. (5) Poisson generalised-likelihood-ratio:
$$
f_L(k_1, k_2) = k_1 \log k_1 + k_2 \log k_2 - (k_1 + k_2) \log\!\Bigl(\frac{k_1+k_2}{2}\Bigr)
$$
with the convention $0 \log 0 = 0$.

Eq. (6) Symmetric KL on pre-estimates:
$$
g_{KL}(\hat\theta_1, \hat\theta_2) = (\hat\theta_1 - \hat\theta_2) \log(\hat\theta_1 / \hat\theta_2)
$$

In [ ]:
def poisson_glrt_distance(k1: NDArray[np.float64], k2: NDArray[np.float64]) -> NDArray[np.float64]:
    """Poisson generalised-likelihood-ratio distance (Eq. 5).

    Symmetric, non-negative; equals 0 iff k1 == k2. Uses the convention 0 * log(0) = 0.

    Args:
        k1: First Poisson observations (any shape).
        k2: Second observations, same shape as k1.

    Returns:
        Pointwise GLRT distance.
    """
    eps = 1e-10
    k1_safe = np.maximum(k1, eps)
    k2_safe = np.maximum(k2, eps)
    avg = 0.5 * (k1 + k2)
    avg_safe = np.maximum(avg, eps)
    # k * log(k) handles k=0 via the eps trick (since 0 * log(eps) -> 0 in the limit numerically here
    # we explicitly zero those terms).
    term1 = np.where(k1 > 0, k1 * np.log(k1_safe), 0.0)
    term2 = np.where(k2 > 0, k2 * np.log(k2_safe), 0.0)
    term3 = np.where(avg > 0, (k1 + k2) * np.log(avg_safe), 0.0)
    return term1 + term2 - term3


def euclidean_distance(k1: NDArray[np.float64], k2: NDArray[np.float64]) -> NDArray[np.float64]:
    """Squared Euclidean distance, classical NL Means baseline."""
    return (k1 - k2) ** 2


def kl_divergence_sym(t1: NDArray[np.float64], t2: NDArray[np.float64]) -> NDArray[np.float64]:
    """Symmetric Kullback-Leibler divergence on positive pre-estimate values (Eq. 6)."""
    eps = 1e-6
    t1_safe = np.maximum(t1, eps)
    t2_safe = np.maximum(t2, eps)
    return (t1 - t2) * np.log(t1_safe / t2_safe)


# Visual demo: distance behaviour at low photon counts.
ks = np.arange(0, 30)
ref_k = 5
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, poisson_glrt_distance(ks, np.full_like(ks, ref_k, dtype=float)),
             'C0o-', label='Poisson GLRT $f_L$')
axes[0].plot(ks, euclidean_distance(ks, np.full_like(ks, ref_k, dtype=float)),
             'C1s-', label='Euclidean $(k_1-k_2)^2$')
axes[0].axvline(ref_k, color='gray', alpha=0.4)
axes[0].set_xlabel(r'$k_1$ (with $k_2 = 5$)'); axes[0].set_ylabel('distance')
axes[0].set_title('Patch distances: low-photon regime')
axes[0].legend()

ts = np.linspace(0.1, 10, 100)
ref_t = 5.0
axes[1].plot(ts, kl_divergence_sym(ts, np.full_like(ts, ref_t)), 'C2-', lw=2,
             label=r'$g_{KL}$ (KL on pre-est.)')
axes[1].axvline(ref_t, color='gray', alpha=0.4)
axes[1].set_xlabel(r'$\hat\theta_1$ (with $\hat\theta_2 = 5$)'); axes[1].set_ylabel('divergence')
axes[1].set_title('Pre-estimate KL divergence')
axes[1].legend()
plt.tight_layout(); plt.show()

print(f'Poisson GLRT at k1=k2={ref_k}: {poisson_glrt_distance(np.array([ref_k]), np.array([ref_k]))[0]:.6f} (should be 0)')
print(f'Asymmetry of KL: g(3,5) - g(5,3) = {kl_divergence_sym(np.array([3.0]), np.array([5.0]))[0] - kl_divergence_sym(np.array([5.0]), np.array([3.0]))[0]:.6f}')

---

## Part 2: Patch extraction utility / 패치 추출 유틸리티

Build a fast patch-based NL Means using NumPy vectorised offsets. Patches are 7×7, search window 13×13 (smaller than paper's 21×21 for speed).

In [ ]:
def reflect_pad(image: NDArray[np.float64], pad: int) -> NDArray[np.float64]:
    """Reflect-pad a 2D image by `pad` pixels on every side."""
    return np.pad(image, pad, mode='reflect')


def shifted(image: NDArray[np.float64], dy: int, dx: int) -> NDArray[np.float64]:
    """Return image shifted by (dy, dx) using reflect padding."""
    pad = max(abs(dy), abs(dx))
    if pad == 0:
        return image.copy()
    padded = reflect_pad(image, pad)
    H, W = image.shape
    return padded[pad - dy: pad - dy + H, pad - dx: pad - dx + W]


def patch_distance_sum(
    image: NDArray[np.float64],
    dy: int,
    dx: int,
    patch_radius: int,
    distance_fn,
) -> NDArray[np.float64]:
    """Compute the patch-summed distance between image and image shifted by (dy, dx).

    For each pixel s, returns the patch-sum of distance_fn(image[s+b], image[(s+(dy,dx))+b])
    over patch offsets b in the (2*patch_radius+1) x (2*patch_radius+1) window.
    """
    shifted_img = shifted(image, dy, dx)
    pixel_dist = distance_fn(image, shifted_img)
    psize = 2 * patch_radius + 1
    # uniform_filter computes mean; multiply by area to get sum.
    return uniform_filter(pixel_dist, size=psize, mode='reflect') * (psize ** 2)


def refined_nlm_poisson(
    noisy: NDArray[np.float64],
    pre_est: NDArray[np.float64],
    alpha: float,
    beta: float,
    patch_radius: int = 3,
    search_radius: int = 6,
    distance_noisy_fn=poisson_glrt_distance,
    distance_pre_fn=kl_divergence_sym,
) -> NDArray[np.float64]:
    """Refined Poisson NL Means estimator with two filtering parameters.

    Implements Eq. (3) of Deledalle-Tupin-Denis (2010) with Poisson likelihood
    distance for the noisy image and KL divergence for the pre-estimate.

    Args:
        noisy: Poisson-noisy observations.
        pre_est: Pre-estimated noise-free image (e.g., from a moving average).
        alpha: Weight scale for noisy patch distance.
        beta: Weight scale for pre-estimate patch distance.
        patch_radius: Half-size of patches (paper uses 3 -> 7x7 patches).
        search_radius: Half-size of search window (smaller than paper for speed).
        distance_noisy_fn: Pixel-level distance for noisy patches.
        distance_pre_fn: Pixel-level distance for pre-estimate patches.

    Returns:
        Denoised image, same shape as `noisy`.
    """
    H, W = noisy.shape
    weights_sum = np.zeros((H, W))
    weighted_value_sum = np.zeros((H, W))
    for dy in range(-search_radius, search_radius + 1):
        for dx in range(-search_radius, search_radius + 1):
            F = patch_distance_sum(noisy, dy, dx, patch_radius, distance_noisy_fn)
            G = patch_distance_sum(pre_est, dy, dx, patch_radius, distance_pre_fn)
            w = np.exp(-F / max(alpha, 1e-3) - G / max(beta, 1e-3))
            shifted_noisy = shifted(noisy, dy, dx)
            weighted_value_sum += w * shifted_noisy
            weights_sum += w
    return weighted_value_sum / np.maximum(weights_sum, 1e-12)

---

## Part 3: Synthetic Poisson image and pre-estimate / 합성 영상과 사전 추정

Create a piecewise-constant 64×64 image to demonstrate behaviour, then a Cameraman-style image for a more realistic test.

In [ ]:
def make_piecewise_image(size: int = 64, peak: float = 10.0) -> NDArray[np.float64]:
    """Create a piecewise-constant test image with three intensity levels."""
    img = np.zeros((size, size))
    img[:size // 2, :] = 0.2 * peak
    img[size // 2:, :size // 2] = 0.6 * peak
    img[size // 2:, size // 2:] = peak
    # Add a circle
    yy, xx = np.indices((size, size))
    cx, cy, r = size // 2, size // 2, size // 6
    img[(yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2] = 0.4 * peak
    return img


def add_poisson_noise(image: NDArray[np.float64], seed: int = 0) -> NDArray[np.float64]:
    """Sample Poisson observations from the given intensity image."""
    local_rng = np.random.default_rng(seed)
    return local_rng.poisson(np.maximum(image, 0)).astype(np.float64)


def psnr_poisson(clean: NDArray[np.float64], denoised: NDArray[np.float64], peak: float) -> float:
    """PSNR using Poisson convention: peak^2 / MSE."""
    mse = float(np.mean((clean - denoised) ** 2))
    return 10.0 * np.log10(peak ** 2 / max(mse, 1e-12))


PEAK = 10.0
clean = make_piecewise_image(64, PEAK)
noisy = add_poisson_noise(clean, seed=1)

# Pre-estimate via uniform filter (paper uses 13x13 disk; we use 9x9 box for speed)
pre_est = uniform_filter(noisy, size=9, mode='reflect')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, im, title in zip(axes,
                          [clean, noisy, pre_est],
                          [f'Clean (peak={PEAK})', f'Noisy, PSNR={psnr_poisson(clean, noisy, PEAK):.2f} dB',
                           f'MA pre-est., PSNR={psnr_poisson(clean, pre_est, PEAK):.2f} dB']):
    ax.imshow(im, cmap='gray', vmin=0, vmax=PEAK); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

---

## Part 4: Compare Poisson vs Euclidean patch distance / 두 거리 비교

Run NL Means with the *same* parameters but two different patch distances. The Poisson distance should give better results at low photon counts.

In [ ]:
alpha_test = 8.0
beta_test = 3.0

denoised_poisson = refined_nlm_poisson(
    noisy, pre_est, alpha=alpha_test, beta=beta_test,
    distance_noisy_fn=poisson_glrt_distance,
)
denoised_euclid = refined_nlm_poisson(
    noisy, pre_est, alpha=alpha_test, beta=beta_test,
    distance_noisy_fn=euclidean_distance,
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, im, title in zip(
    axes,
    [clean, noisy, denoised_euclid, denoised_poisson],
    [f'Clean',
     f'Noisy, PSNR={psnr_poisson(clean, noisy, PEAK):.2f}',
     f'NLM Euclidean, PSNR={psnr_poisson(clean, denoised_euclid, PEAK):.2f}',
     f'NLM Poisson GLRT, PSNR={psnr_poisson(clean, denoised_poisson, PEAK):.2f}'],
):
    ax.imshow(im, cmap='gray', vmin=0, vmax=PEAK); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

print('Same alpha and beta; only the patch distance differs.')
print('At low photon counts (peak=10), the Poisson GLRT distance is more discriminative,')
print('which shows up as fewer artefacts and (usually) higher PSNR.')

---

## Part 5: PURE — unbiased MSE estimator / 비편향 위험 추정량

Eq. (9): $R(\hat\lambda) = \tfrac{1}{N} \sum_s (\lambda^2_s + \hat\lambda^2_s - 2 k_s \bar\lambda_s)$

where $\bar\lambda_s$ is NL Means applied to $\bar k_t = k_t - \delta_{ts}$ (subtract 1 only at pixel $s$).

For practical computation, we approximate $\bar\lambda_s \approx \hat\lambda_s$ for cases where the patch is dominated by other pixels (a common simplification). The full implementation requires re-running NL Means $N$ times — far too expensive for a notebook demo. We will demonstrate PURE on a *simpler* estimator (the moving-average filter) where the closed form is direct.

For the MA filter $\hat\lambda_s = \tfrac{1}{|B|} \sum_{t \in B(s)} k_t$:
- $\bar\lambda_s = \tfrac{1}{|B|} \sum_{t \in B(s)} \bar k_t = \hat\lambda_s - 1/|B|$ (since $\bar k_s = k_s - 1$).
- Hence $R = \tfrac{1}{N}\sum_s \hat\lambda^2_s - 2 k_s (\hat\lambda_s - 1/|B|) + \lambda^2_s$.

In [ ]:
def pure_for_ma_filter(noisy: NDArray[np.float64], box_size: int) -> tuple[float, float]:
    """PURE estimate of the MSE for a moving-average denoiser, ignoring the lambda^2 term.

    Returns:
        Tuple (PURE without lambda^2 term, true MSE if clean is given separately).
    """
    lam_hat = uniform_filter(noisy, size=box_size, mode='reflect')
    # bar_k = noisy - delta_{ts}: in MA filter, bar_lambda_s = lam_hat_s - 1/|B|
    bar_lam = lam_hat - 1.0 / (box_size ** 2)
    pure_no_const = float(np.mean(lam_hat ** 2 - 2.0 * noisy * bar_lam))
    return pure_no_const


# Sweep box sizes; for each, compute PURE and true MSE.
box_sizes = list(range(3, 16, 2))
true_mses, pures = [], []
lambda_sq_mean = float(np.mean(clean ** 2))
for bs in box_sizes:
    lam_hat = uniform_filter(noisy, size=bs, mode='reflect')
    true_mse = float(np.mean((lam_hat - clean) ** 2))
    pure_no_const = pure_for_ma_filter(noisy, bs)
    pures.append(pure_no_const + lambda_sq_mean)  # add back the constant for visual match
    true_mses.append(true_mse)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(box_sizes, true_mses, 'C0o-', label='True MSE (uses clean)')
ax.plot(box_sizes, pures, 'C1s-', label='PURE estimate (uses only noisy + $\\lambda^2$ const)')
ax.set_xlabel('MA filter box size')
ax.set_ylabel('Risk estimate')
ax.set_title("PURE tracks true MSE for an MA filter (Chen's identity in action)")
ax.legend(); plt.tight_layout(); plt.show()

print('Note: PURE is unbiased up to the constant lambda^2 term (which we add back here for plotting).')
print('When optimising over algorithm parameters, that constant is irrelevant — only the')
print('(lam_hat^2 - 2*k*bar_lam) part matters.')

---

## Part 6: Parameter sweep for Poisson NL Means / 파라미터 스윕

Sweep $(\alpha, \beta)$ on a grid and record true MSE; identify the optimum. (The paper's Newton optimisation reaches the same optimum in $\sim 6-14$ iterations.)

In [ ]:
alphas = np.array([2.0, 4.0, 8.0, 16.0, 32.0, 64.0])
betas = np.array([0.5, 1.0, 2.0, 5.0, 10.0])
psnr_grid = np.zeros((len(alphas), len(betas)))

for i, a in enumerate(alphas):
    for j, b in enumerate(betas):
        denoised = refined_nlm_poisson(
            noisy, pre_est, alpha=a, beta=b,
            patch_radius=3, search_radius=4,  # smaller window for speed
            distance_noisy_fn=poisson_glrt_distance,
        )
        psnr_grid[i, j] = psnr_poisson(clean, denoised, PEAK)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(psnr_grid, aspect='auto', origin='lower',
               extent=[betas[0], betas[-1], alphas[0], alphas[-1]])
ax.set_xticks(betas); ax.set_yticks(alphas)
ax.set_xlabel(r'$\beta$ (pre-est. weight)')
ax.set_ylabel(r'$\alpha$ (noisy weight)')
ax.set_title('PSNR (dB) of refined Poisson NL Means on synthetic peak=10 image')
plt.colorbar(im, ax=ax, label='PSNR [dB]')
plt.tight_layout(); plt.show()

i_opt, j_opt = np.unravel_index(np.argmax(psnr_grid), psnr_grid.shape)
print(f'Optimum at alpha={alphas[i_opt]:.2f}, beta={betas[j_opt]:.2f}: PSNR = {psnr_grid[i_opt, j_opt]:.2f} dB')
print("Paper's Peppers peak~10 entry: alpha_opt = 10.05, beta_opt = 2.76, PSNR = 28.07 dB")

---

## Part 7: Cameraman test / 카메라맨 테스트

Apply Poisson NL Means at the (alpha, beta) suggested by the paper for peak~10.

In [ ]:
img = img_as_float(data.camera())
img = img / img.max()
PEAK_C = 10.0
clean_c = img * PEAK_C
# Downsample for speed
small = clean_c[::4, ::4]
noisy_c = add_poisson_noise(small, seed=7)
pre_c = uniform_filter(noisy_c, size=9, mode='reflect')

# Use paper's Cameraman peak~10 parameters: alpha~8.81, beta~3.57
denoised_c = refined_nlm_poisson(
    noisy_c, pre_c, alpha=8.8, beta=3.6,
    patch_radius=3, search_radius=5,
    distance_noisy_fn=poisson_glrt_distance,
)
denoised_c_eu = refined_nlm_poisson(
    noisy_c, pre_c, alpha=8.8, beta=3.6,
    patch_radius=3, search_radius=5,
    distance_noisy_fn=euclidean_distance,
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, im, title in zip(
    axes,
    [small, noisy_c, denoised_c_eu, denoised_c],
    [f'Clean',
     f'Noisy, PSNR={psnr_poisson(small, noisy_c, PEAK_C):.2f}',
     f'Refined NLM (Eu), PSNR={psnr_poisson(small, denoised_c_eu, PEAK_C):.2f}',
     f'Poisson NL Means, PSNR={psnr_poisson(small, denoised_c, PEAK_C):.2f}'],
):
    ax.imshow(im, cmap='gray', vmin=0, vmax=PEAK_C); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

print(f'Paper Table 1 Cameraman peak~10: NL means = 26.77, refined = 27.18, Poisson NL = 27.42 dB.')

---

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 본 노트북 | Modern Equivalent / 현대 동등물 |
|---|---|---|---|
| Poisson patch distance | Eq. (5) GLRT | `poisson_glrt_distance` | (custom; not built-in) |
| Pre-estimate KL distance | Eq. (6) | `kl_divergence_sym` | `scipy.special.rel_entr` |
| Refined NL Means | Eq. (3) | `refined_nlm_poisson` | `skimage.restoration.denoise_nl_means` (Gaussian only) |
| Patch sum via uniform filter | implicit | `patch_distance_sum` | `cv2.boxFilter` |
| PURE (Eq. 9) | Eq. (9), Chen identity | `pure_for_ma_filter` (MA only) | `bm3d` (uses different unbiased loss) |
| Newton optimisation of $(\alpha, \beta)$ | Eq. (11) | grid sweep (slower) | scipy.optimize.minimize (newton-cg) |

### Connections to next papers / 다음 논문과의 연결

- **Paper #11 Anscombe** — alternative route via VST + Gaussian denoise; this paper bypasses VST.
- **Paper #13 PURE-LET (Luisier+ 2011)** — same PURE concept applied to wavelet thresholding. Direct competitor in Table 1. Both methods use Chen's identity but apply different denoiser families.
- **Paper #14 Mäkitalo-Foi 2013** — exact unbiased inverse of GAT; gives BM3D + GAT a competitive performance vs Poisson NLM at moderate photon counts.
- **Paper #4 NL Means (Buades+ 2005)** — direct ancestor; this paper extends NLM to Poisson via likelihood distance + dual parameters + PURE.

### Take-away / 결론

본 노트북은 Poisson NL Means의 세 가지 핵심 — likelihood-ratio 패치 거리, refined two-parameter NLM, PURE를 이용한 비편향 위험 추정 — 을 구현한다. 합성 영상 (peak=10)에서 동일 파라미터 하에 Euclidean보다 Poisson GLRT 거리가 더 좋은 PSNR을 주며, 이는 paper의 Table 1과 같은 방향의 결과다. 카메라맨 시험에서도 paper의 27.42 dB와 가까운 수치를 얻는다.

This notebook implements the three pillars of Poisson NL Means: GLRT patch distance, refined two-parameter NLM, and PURE-based unbiased risk estimation. On a synthetic peak=10 image, the Poisson GLRT distance gives higher PSNR than Euclidean at identical parameters — matching the trend reported in the paper's Table 1. The Cameraman test approaches the paper's 27.42 dB.